## Introduction

This notebook contains a simplified and shortened analysis, that matches what was done in the paper "_Measurements of Forbush decrease events at the center of the South Atlantic Magnetic Anomaly with Muon detectors_" by J. Molina et. al.

The code is kept simple and procedural, so it can be more easily followed by anyone without expertise in production style code.

In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.express as px
from scipy import stats

## Load Data

This repository contains data from the FIUNA (Faculty of Engineering National University of Asunción)
muon detector, plus temperature and pressure sensors.

All timestamps are in UTC time. The muon counts are given per minute. The observations
of temperature and pressure had varying cadences, so for each minute of muon data the nearest-in-time value of each other parameter has been added, for ease of use.
Please note that this is the post-processed data, after detector down-time periods have been removed and some additional cleaning has been applied.

Data from two analysis periods (May\~Jun and Sep\~Oct 2024) are available.

The Dst data, and the neutron data from the Mexico NMDB (Neutron Monitor DataBase), are also included in the data folder for convenience (although these have a one hour cadence rather than a one minute cadence; links and references in the paper).

In [6]:
#data_folder = 'data/first_period/'
data_folder = 'data/second_period/'

df = pd.read_csv(data_folder + 'muons_sep_octcorrected.csv', index_col='datetime', parse_dates=['datetime'])
dst = pd.read_csv(data_folder + 'dst.csv', index_col='timestamp', parse_dates=['timestamp'])
neutrons = pd.read_csv('data/first_period/neutrons.csv', index_col='timestamp', parse_dates=['timestamp'])

## Temperature and Pressure Corrections

As discussed in the paper, our detection equipment is affected by local (laboratory) temperature, and the impingent muon flux is affected by atmospheric temperature and pressure (compared to the primary flux at the upper atmosphere, which is what we wish to reconstruct).

As these variables are correlated, we perform a multivariate linear fit to reconstruct the estimated true muon flux.

In [7]:
# Use statsmodels library for the fit.
muon_col = 'muons'
fit_cols = ['lab_temp','external_temp','external_pressure']

mod = sm.OLS(df[muon_col], sm.add_constant(df[fit_cols].values))
fit = mod.fit()
errors = fit.summary2().tables[1]['Std.Err.'][1:].values
const = fit.params.values[0]
params = fit.params.values[1:]
means = df.mean()[fit_cols].values

# Calculate the necessary correction based on this fit.
df['correction'] = df.apply(lambda r: np.sum((means - r[fit_cols].values) * params), axis=1)

# Use 2 * errors for 95% confidence interval
df['correction_err'] = df.apply(lambda r: np.sqrt(np.sum(
    ((means - r[fit_cols].values) * errors * 2) ** 2
)), axis=1)

# Apply the correction.
df['muons_corrected'] = df[muon_col] + df['correction']

# Show the results
params, errors, means

KeyError: "['external_temp', 'external_pressure'] not in index"

## Hourly Resampling

This data can be examined at any point, for example Plotly provides an interactive (zoomable) plot:

In [ ]:
px.line(df[['muons','muons_corrected']])

h:\Anaconda\envs\IA\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning: The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result
  v = v.dt.to_pydatetime()


## Statistical Testing

For statistical testing against the Dst data, which is only available hourly, we will need muon data with an hourly cadence. Some of the per-minute muon data $\mu_{m}$ contains small gaps of 1 minute, so we do some minor rescaling to the hourly counts, by using ${\bar{\mu}_{m}} \times 60$ instead of $\sum\mu_{m}$.

In [ ]:
muons_per_hour = df[['muons_corrected']].resample('h').mean() * 60
px.line(muons_per_hour)

h:\Anaconda\envs\IA\lib\site-packages\_plotly_utils\basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



We then perform a TTS (Truncated Time Shift) test, to determine if we can reject the null hypothesis of uncorrelated data. First define our two input series, which must share an identical and continuous time index.

In [9]:
# Align on continuous time index using Pandas join; impute gaps
aligned_data = df.join(dst).interpolate(method='linear')
x = aligned_data['dst'].copy()
y = aligned_data['muons'].copy()

# For the TTS test to apply, at least one input series must pass an Augmented Dickey-Fuller test for stationarity.
# Here the p-values are printed, at least one of which should be low.
from statsmodels.tsa.stattools import adfuller
adfuller(x.values)[1], adfuller(y.values)[1]

(0.0002121683652090854, 0.0009704453235825364)

The TTS parameter $r$ (which controls the amount of data that is truncated from the edge of one time series, to create a sliding window from the second time series), must be set depending on the length of data available and desired granularity of the $u$-statistic. Since our data is short and we know the Forbush decrease occurred early in the dataset, we must chose a relatively small value of $r=50$. This gives a coarse granularity for the $u$-statistic, but is still enough to successfully reject the null hypothesis, if the data do support such a conclusion.

The second parameter $l$, which represents the expected time-lag of the correlation between the signals, must also be set without reference to the data itself (e.g. based on external physical considerations, and fixed before applying the test). As referenced in our paper, the expected time lag between the Dst and muon measurements is approximately +2 hours at our location, so we set $l=2$.

Finally, we generate a list of $\tau$ parameters $\tau_0, \ldots, \tau_{2r+1}$ which are Pearson-$r$ values of the shifted time series.

In [10]:
# TTS test parameters
r = 50
l = 2

# Perform the test; 
taus = []
x_trunc = x[r - l : - (r + l)]
for delta in range(0, r*2 + 1):
    y_trunc = y[delta : len(x_trunc)+delta]
    tau = stats.pearsonr(x_trunc.values, y_trunc.values).statistic
    taus.append(np.abs(tau))

# Visualise, if desired


In [11]:
px.line(taus)

If the two series are correlated best at the unshifted position ($\delta = r$), then $\tau_r$ should have the highest value (or close to it). The TTS test statistic is:

$u = \frac{B}{r+1}$

where $B$ is the rank (integer position) of $\tau_r$ in the list of $\tau$ parameters.

In other words, the unshifted signal should be amongst the $n$-best correlations (depending on the desired accuracy of the test; in our paper we set $u=0.1$ as the significance threshold).

In [12]:
r=50
# Sort the taus by descending value
t = pd.Series(taus).sort_values(ascending=False)

# Get the rank (position) of tau_r
B = t.index.get_loc(r) + 1

# Get the u-statistic
u = B / (r+1)

print(f"B: {B}, u: {u:.4f}")

B: 2, u: 0.0392


In [ ]:
B

4